# AURA Disease Prediction: Exploratory Data Analysis (EDA)
### Version 2 - Target Disease Groupings:
1. **Group 1 (Thyroid Dysfunction):** Hypothyroidism, Hyperthyroidism, Subclinical Thyroid Dysfunction
2. **Group 2 (Nutritional Deficiencies):** Iron Deficiency Anemia, Vitamin D Deficiency, Vitamin B12 Deficiency, Folate Deficiency

---

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Plot styling
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

RAW_DIR = Path("../datasets/raw")
PROCESSED_DIR = Path("../datasets/processed")
print("Raw directory exists:", RAW_DIR.exists())
print("Processed directory exists:", PROCESSED_DIR.exists())

## 1. Group 1: Thyroid Disease Distribution & Lab Profiles
- Normal (0)
- Hypothyroidism (1): TSH > 4.5 mIU/L AND FT4 < 0.8 ng/dL
- Hyperthyroidism (2): TSH < 0.4 mIU/L AND FT4 > 1.8 ng/dL
- Subclinical Thyroid (3): TSH > 4.5 mIU/L AND FT4 normal

In [ ]:
thyroid_path = RAW_DIR / "thyroid_raw.csv"
if thyroid_path.exists():
    df_thyroid = pd.read_csv(thyroid_path)
    print("Thyroid Raw Shape:", df_thyroid.shape)
    display(df_thyroid.head())
    
    # Distribution of TSH & FT4
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    if "TSH" in df_thyroid.columns:
        sns.histplot(df_thyroid["TSH"], bins=30, kde=True, ax=axes[0], color="royalblue")
        axes[0].set_title("Serum TSH Distribution (mIU/L)")
    if "FT4" in df_thyroid.columns:
        sns.histplot(df_thyroid["FT4"], bins=30, kde=True, ax=axes[1], color="crimson")
        axes[1].set_title("Serum Free T4 Distribution (ng/dL)")
    plt.tight_layout()
    plt.show()
else:
    print("Thyroid dataset not found in raw/. Run datasets/download_all.py --generate-samples first.")

## 2. Group 2: Iron Deficiency Anemia (Hematology Panel)
- Hemoglobin (Hb) & Mean Corpuscular Volume (MCV)
- WHO Thresholds: Hb < 12/13 g/dL AND MCV < 80 fL

In [ ]:
anemia_path = RAW_DIR / "anemia_raw.csv"
if anemia_path.exists():
    df_anemia = pd.read_csv(anemia_path)
    print("Anemia Raw Shape:", df_anemia.shape)
    display(df_anemia.describe())
    
    # Scatter plot: Hb vs MCV microcytosis
    plt.figure(figsize=(9, 6))
    target_col = "Diagnosis" if "Diagnosis" in df_anemia.columns else "target"
    sns.scatterplot(data=df_anemia, x="Hemoglobin", y="MCV", hue=target_col, palette="Set1", alpha=0.8)
    plt.axvline(12.0, color="red", linestyle="--", label="Hb Cutoff (12 g/dL)")
    plt.axhline(80.0, color="orange", linestyle="--", label="MCV Microcytosis (80 fL)")
    plt.title("Hemoglobin vs. MCV: Iron Deficiency Microcytosis Quadrant")
    plt.legend()
    plt.show()
else:
    print("Anemia raw file not found.")

## 3. Group 2: Vitamin D Deficiency Cohort
- 25(OH)D < 20 ng/mL Cutoff (Endocrine Society)

In [ ]:
vitd_path = RAW_DIR / "vitamin_d_raw.csv"
if vitd_path.exists():
    df_vitd = pd.read_csv(vitd_path)
    print("Vitamin D Cohort Shape:", df_vitd.shape)
    vitd_col = next((c for c in df_vitd.columns if "vid" in c.lower() or "25oh" in c.lower()), None)
    if vitd_col:
        plt.figure(figsize=(9, 5))
        sns.histplot(df_vitd[vitd_col], bins=35, kde=True, color="goldenrod")
        plt.axvline(20.0, color="red", linestyle="--", label="Deficiency Cutoff (< 20 ng/mL)")
        plt.title("Serum 25-hydroxyvitamin D Distribution")
        plt.legend()
        plt.show()
else:
    print("Vitamin D raw file not found.")

## 4. Group 2: Vitamin B12 & Folate Laboratory Cohort
- Serum B12 < 200 pg/mL
- Serum Folate < 4.0 ng/mL

In [ ]:
b12_path = RAW_DIR / "b12_folate_raw.csv"
if b12_path.exists():
    df_b12 = pd.read_csv(b12_path)
    print("B12 & Folate Raw Shape:", df_b12.shape)
    b12_col = next((c for c in df_b12.columns if "b12" in c.lower()), None)
    folate_col = next((c for c in df_b12.columns if "folate" in c.lower() or "fot" in c.lower()), None)
    
    if b12_col and folate_col:
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        sns.histplot(df_b12[b12_col], bins=30, kde=True, ax=axes[0], color="mediumpurple")
        axes[0].axvline(200.0, color="red", linestyle="--", label="B12 < 200 pg/mL")
        axes[0].set_title("Serum B12 Distribution")
        axes[0].legend()
        
        sns.histplot(df_b12[folate_col], bins=30, kde=True, ax=axes[1], color="seagreen")
        axes[1].axvline(4.0, color="red", linestyle="--", label="Folate < 4 ng/mL")
        axes[1].set_title("Serum Folate Distribution")
        axes[1].legend()
        plt.tight_layout()
        plt.show()
else:
    print("B12 & Folate dataset not found.")

## 5. Correlation Matrix & Clinical Collinearity Check

In [ ]:
if anemia_path.exists():
    num_df = df_anemia.select_dtypes(include=[np.number])
    plt.figure(figsize=(8, 6))
    sns.heatmap(num_df.corr(), annot=True, cmap="coolwarm", fmt=".2f", cbar=True)
    plt.title("Correlation Heatmap: Hematological Indices")
    plt.show()